# 🦜🔗 LangChain — Pilar 3: Orchestration & Composition

> **Versões:** `langchain-core==1.4.0` | `langchain-openai==1.1.10` | `langchain-community==1.0.0` | Maio/2026

---

## O que é Orchestration & Composition?

Unificamos o conhecimento prévio dos pilares fundamentais (**Model I/O** e **RAG**) para criar sistemas estruturados, dotados de memória de longo prazo e capazes de raciocínio de execução dinâmico por meio de **Agentes Autônomos**.

| Componente | O que faz | Analogia |
|---|---|---|
| **Chains** | Sequências encadeadas e estáticas de ações lógicas (Passo A → B) | Receita de bolo |
| **Memory** | Mantém a retenção de contexto do histórico de diálogos | Bloco de notas persistente |
| **Agents** | Decide em tempo real qual ação tomar baseando-se no input recebido | Chef improvisando pratos |
| **Tools** | Funções programáticas executadas pelos Agentes para interação externa | Utensílios de cozinha |

---

## ⚙️ Instalação

In [ ]:
%pip install -qU langchain-core==1.4.0 langchain-openai==1.1.10 langchain-community==1.0.0 langchain sympy python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    raise EnvironmentError(
        'OPENAI_API_KEY não encontrada.\n'
        'Crie um arquivo .env na raiz do projeto com:\n'
        'OPENAI_API_KEY=sk-...'
    )

# Necessário para hub.pull() na seção 4.1 (ReAct Agent)
if not os.getenv('LANGCHAIN_API_KEY'):
    print('Aviso: LANGCHAIN_API_KEY não definida — hub.pull() na seção 4.1 não funcionará.')
    print('Adicione ao .env: LANGCHAIN_API_KEY=ls__...')
    print('Obtenha sua chave em: https://smith.langchain.com/')

print('Ambiente configurado!')

---
## 1. ⛓️ Chains — Sequências fixas com LCEL

O ecossistema LangChain consolidou o uso de componentes unificados via **LCEL** recorrendo ao operador de encadeamento `|`. Classes legadas como `LLMChain` e `SequentialChain` foram descontinuadas.

### 1.1 Chain básica

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0.7)

# Chain 1: Gera um título
prompt_titulo = ChatPromptTemplate.from_template(
    'Crie um título criativo para um artigo sobre: {tema}. Responda só com o título.'
)

chain_titulo = prompt_titulo | llm | StrOutputParser()

titulo = chain_titulo.invoke({'tema': 'inteligência artificial no agronegócio'})
print('Título gerado:', titulo)

### 1.2 Chains encadeadas (Sequential)

In [ ]:
from langchain_core.runnables import RunnableLambda

prompt_resumo = ChatPromptTemplate.from_template(
    'Escreva um parágrafo de introdução para um artigo com o título: "{titulo}"'
)

chain_resumo = prompt_resumo | llm | StrOutputParser()

chain_completa = chain_titulo | RunnableLambda(lambda t: {'titulo': t}) | chain_resumo

resultado = chain_completa.invoke({'tema': 'machine learning em saúde'})
print(resultado)

### 1.3 RunnableParallel — Executando fluxos em paralelo

In [ ]:
from langchain_core.runnables import RunnableParallel

# Definimos chains independentes que herdam a mesma chamada de input de forma assíncrona/paralela
chain_pro = (
    ChatPromptTemplate.from_template('Liste 2 vantagens de: {tecnologia}') 
    | llm | StrOutputParser()
)

chain_contra = (
    ChatPromptTemplate.from_template('Liste 2 desvantagens de: {tecnologia}')
    | llm | StrOutputParser()
)

chain_exemplo = (
    ChatPromptTemplate.from_template('Dê 1 caso de uso real de: {tecnologia}')
    | llm | StrOutputParser()
)

# Paralelização lógica via RunnableParallel
chain_paralela = RunnableParallel(
    vantagens=chain_pro,
    desvantagens=chain_contra,
    caso_de_uso=chain_exemplo
)

resultado = chain_paralela.invoke({'tecnologia': 'blockchain'})

print('=== ANÁLISE DE BLOCKCHAIN ===')
print('\n✅ VANTAGENS:')
print(resultado['vantagens'])
print('\n❌ DESVANTAGENS:')
print(resultado['desvantagens'])
print('\n💡 CASO DE USO:')
print(resultado['caso_de_uso'])

### 1.4 RunnablePassthrough — Preservando variáveis

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Preserva o dicionário ou valor de input intacto para reutilizações posteriores
chain_com_passthrough = RunnableParallel(
    pergunta_original=RunnablePassthrough(),
    resposta=(
        ChatPromptTemplate.from_template('Responda em 1 frase: {input}')
        | llm 
        | StrOutputParser()
    )
)

resultado = chain_com_passthrough.invoke({'input': 'O que é LangChain?'})
print('Pergunta original:', resultado['pergunta_original'])
print('Resposta:', resultado['resposta'])

---
## 2. 🧠 Memory — Contexto histórico da conversa

Abordagens de produção empregam a unificação nativa de instâncias de controle de histórico por sessões estruturadas via `ChatMessageHistory` combinadas ao espaço dinâmico reservado em prompt pelo objeto `MessagesPlaceholder`.

### 2.1 Gerenciando histórico manualmente

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0.7)

prompt_chat = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente amigável. Responda em português.'),
    MessagesPlaceholder(variable_name='historico'),
    ('human', '{mensagem}')
])

chain_chat = prompt_chat | llm | StrOutputParser()

historico = []

def conversar(mensagem: str) -> str:
    resposta = chain_chat.invoke({
        'historico': historico,
        'mensagem': mensagem
    })
    historico.append(HumanMessage(content=mensagem))
    historico.append(AIMessage(content=resposta))
    return resposta

print('Turno 1:')
r1 = conversar('Olá! Meu nome é Ana e sou engenheira de dados.')
print('Bot:', r1)

print('\nTurno 2:')
r2 = conversar('Qual é a minha profissão?')
print('Bot:', r2)

### 2.2 Gerenciamento automático de sessão por ID (`RunnableWithMessageHistory`)

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt_sessao = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente. Responda em português.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

chain_base = prompt_sessao | llm | StrOutputParser()

chain_com_memoria = RunnableWithMessageHistory(
    chain_base,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history'
)

config_user1 = {'configurable': {'session_id': 'usuario_1'}}
config_user2 = {'configurable': {'session_id': 'usuario_2'}}

r = chain_com_memoria.invoke({'input': 'Meu nome é João e moro em São Paulo.'}, config=config_user1)
print('User1, turno 1:', r)

r = chain_com_memoria.invoke({'input': 'Meu nome é Maria e moro em Curitiba.'}, config=config_user2)
print('User2, turno 1:', r)

r = chain_com_memoria.invoke({'input': 'Qual é o meu nome e onde moro?'}, config=config_user1)
print('\nUser1, turno 2:', r)

---
## 3. 🔨 Tools — As mãos do agente

Tools são funções Python anotadas com `@tool` que os agentes podem invocar para interagir com o mundo externo: calcular valores, consultar APIs, buscar dados, etc.

### Segurança: avaliação de expressões matemáticas

Receber expressões matemáticas como texto do usuário é um caso de uso comum, mas exige atenção. A função nativa `eval()` do Python executa **qualquer código arbitrário**, tornando-se um vetor crítico de **Remote Code Execution (RCE)** se exposta.

| Abordagem | Risco | Indicação |
|---|---|---|
| `eval(expressao)` | Crítico — executa código Python arbitrário | Nunca em produção |
| `sympy.sympify()` | Seguro — parsing matemático sem execução de código | **Recomendado** |
| `numexpr` | Seguro e de alta performance numérica | Produção com grandes volumes |

A ferramenta `calcular` abaixo usa `sympy.sympify()`: ele constrói a expressão como uma árvore matemática e a avalia de forma isolada do runtime Python, sem risco de injeção de código.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import sympy

@tool
def obter_data_hora() -> str:
    """Retorna a data e hora atual."""
    return datetime.now().strftime('Data: %d/%m/%Y | Hora: %H:%M:%S')

@tool
def calcular(expressao: str) -> str:
    """Avalia expressões matemáticas de forma segura. Suporta +, -, *, /, ** e funções como sqrt, sin, cos, log."""
    try:
        resultado = sympy.sympify(expressao, evaluate=True)
        return f'Resultado: {float(resultado):.6g}'
    except sympy.SympifyError:
        return 'Erro: expressão matemática inválida.'
    except Exception as e:
        return f'Erro: {type(e).__name__}'

@tool
def celsius_para_fahrenheit(celsius: float) -> str:
    """Converte uma temperatura de graus Celsius para Fahrenheit."""
    fahrenheit = (celsius * 9/5) + 32
    return f'{celsius}°C = {fahrenheit:.1f}°F'

print(calcular.invoke({'expressao': '150 + 25 - 10'}))
print(calcular.invoke({'expressao': 'sqrt(144)'}))

### 3.2 Tool com Schema Pydantic Personalizado

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

class BuscaInput(BaseModel):
    query: str = Field(description='O texto a ser pesquisado')
    max_resultados: int = Field(default=3, description='Número máximo de resultados (1-10)')
    categoria: Literal['noticias', 'academico', 'geral'] = Field(
        default='geral',
        description='Categoria da busca'
    )

@tool(args_schema=BuscaInput)
def buscar_na_web(query: str, max_resultados: int = 3, categoria: str = 'geral') -> str:
    """Simula de forma controlada uma busca na web externa e retorna os principais resultados encontrados."""
    return f'[SIMULAÇÃO] Busca por "{query}" na categoria "{categoria}": {max_resultados} resultado(s) encontrado(s).'

print(buscar_na_web.invoke({
    'query': 'LangChain agents 2026',
    'max_resultados': 5,
    'categoria': 'academico'
}))

---
## 4. 🤖 Agents — Raciocínio dinâmico

Diferente das cadeias estáticas fixas, agentes processam o fluxo em laços iterativos de tomada de decisões autônomas. Apresentamos o modelo clássico conceitual ReAct e a implementação recomendada por Function Calling Nativo.

### 4.1 Arquitetura ReAct Clássica

> **Pré-requisito:** `hub.pull()` requer a variável `LANGCHAIN_API_KEY` configurada no `.env`. Obtenha sua chave em [smith.langchain.com](https://smith.langchain.com/).

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_openai import ChatOpenAI

ferramentas = [obter_data_hora, calcular, celsius_para_fahrenheit, buscar_na_web]
llm = ChatOpenAI(model='gpt-4o', temperature=0)

prompt_react = hub.pull('hwchase17/react')

agente = create_react_agent(
    llm=llm,
    tools=ferramentas,
    prompt=prompt_react
)

executor = AgentExecutor(
    agent=agente,
    tools=ferramentas,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

In [ ]:
resultado = executor.invoke({'input': 'Que dia é hoje?'})
print('\n✅ Resposta final:', resultado['output'])

### 4.2 Abordagem Moderna: Tool Calling Agent (Recomendado)

Modelos de fronteira atuais (como os das famílias GPT, Claude e Gemini) possuem inteligência de mapeamento e ativação de assinaturas de código nativas por APIs de chamadas de função, tornando o uso de prompts textuais interpretativos ReAct obsoletos e suscetíveis a falhas de sintaxe.

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0)

prompt_tool_calling = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente inteligente. Use as ferramentas disponíveis quando necessário.'),
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad')  # bloco necessário para controle interno de iterações do agente
])

agente_tc = create_tool_calling_agent(
    llm=llm,
    tools=ferramentas,
    prompt=prompt_tool_calling
)

executor_tc = AgentExecutor(
    agent=agente_tc,
    tools=ferramentas,
    verbose=False,
    max_iterations=5
)

resultado = executor_tc.invoke({
    'input': 'Qual é o resultado de 150 + 25? E quantos graus Fahrenheit é 25°C?'
})

print('Resposta Estruturada Nativa:', resultado['output'])

---
## 5. 🏗️ Agente Completo de Produção: Memória + Tools

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o', temperature=0.3)
ferramentas = [obter_data_hora, calcular, celsius_para_fahrenheit]

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente pessoal útil e amigável. Responda em português. Use as ferramentas quando necessário.'),
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad')
])

agente = create_tool_calling_agent(llm, ferramentas, prompt)
executor = AgentExecutor(agent=agente, tools=ferramentas, verbose=False)

store_sessoes = {}

def get_history(session_id: str):
    if session_id not in store_sessoes:
        store_sessoes[session_id] = InMemoryChatMessageHistory()
    return store_sessoes[session_id]

agente_com_memoria = RunnableWithMessageHistory(
    executor,
    get_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)

config = {'configurable': {'session_id': 'sessao_demo'}}

perguntas = [
    'Olá! Me chamo Lucas e estou desenvolvendo um app de saúde.',
    'Que horas são agora?',
    'A temperatura corporal normal é 37°C. Quanto é isso em Fahrenheit?',
    'Lembrando do meu projeto de app, qual é o meu nome?'
]

print('=== AGENTE COMPLETO COM MEMÓRIA E TOOLS ===\n')
for pergunta in perguntas:
    print(f'👤 Usuário: {pergunta}')
    resp = agente_com_memoria.invoke({'input': pergunta}, config=config)
    print(f'🤖 Agente: {resp["output"]}')
    print()

---
## 📚 Resumo Final — Os 3 Pilares do LangChain

```
┌──────────────────────────────────────────────────────────┐
│                    LANGCHAIN                             │
├──────────────┬───────────────────┬──────────────────────┤
│  Model I/O   │ Data Connection   │  Orchestration &     │
│              │      / RAG        │  Composition         │
├──────────────┼───────────────────┼──────────────────────┤
│ • ChatOpenAI │ • TextLoader      │ • LCEL (|)           │
│ • ChatPrompt │ • RecursiveSplit  │ • RunnableParallel   │
│ • StrParser  │ • OpenAIEmbeds    │ • Memory (History)   │
│ • PydanticP  │ • FAISS / Chroma  │ • @tool decorator    │
│              │ • as_retriever()  │ • AgentExecutor      │
└──────────────┴───────────────────┴──────────────────────┘
```

### Próximos passos no ecossistema LangChain:

- **LangGraph** — Recomendado para fluxos complexos baseados em **grafos de estados com ciclos lógicos** e múltiplos agentes colaborativos.
- **LangSmith** — Plataforma corporativa para **observabilidade, monitoramento e tracing** em tempo real de latência de prompts e consumo de tokens.
- **LangServe** — Facilita a exposição de suas chains e runnables como microsserviços e rotas de APIs REST de produção prontas para consumo.